In [1]:
import tkinter as tk
from tkinter import filedialog
import sys
import pandas as pd
import numpy as np

# --- GUI Logic Start ---
# Create a hidden root window
root = tk.Tk()
root.withdraw()

print("Please select the Master File (PO Part Accounts)...")
master_path = filedialog.askopenfilename(
    title="Select Master File (PO Part Accounts)",
    filetypes=[("Excel/CSV Files", "*.xlsx *.xls *.csv")]
)

if not master_path:
    print("No Master file selected. Exiting.")
    sys.exit()

print("Please select the Equipment Import File (Multiple Occurrences)...")
import_path = filedialog.askopenfilename(
    title="Select Equipment Import File",
    filetypes=[("Excel/CSV Files", "*.xlsx *.xls *.csv")]
)

if not import_path:
    print("No Import file selected. Exiting.")
    sys.exit()

print(f"\nProcessing:\nMaster: {master_path}\nImport: {import_path}\n")
# --- GUI Logic End ---

# --- Your Processing Logic Below ---

# Load files (Using the paths selected above)
# Detect file type to use correct loader
if master_path.endswith('.csv'):
    df_master = pd.read_csv(master_path)
else:
    df_master = pd.read_excel(master_path)

if import_path.endswith('.csv'):
    df_import = pd.read_csv(import_path)
else:
    # Assuming the tab name is 'Multiple Occurrences' based on your previous prompt
    # If the user selects the raw Excel, we might need to specify the sheet_name
    try:
        df_import = pd.read_excel(import_path, sheet_name='Multiple Occurrences')
    except:
        # Fallback if specific sheet not found, read first sheet
        df_import = pd.read_excel(import_path)

# Rename columns for consistency
# Note: Ensure these map to your actual file columns
df_master = df_master.rename(columns={
    'PO #': 'PO', 'Account #': 'Account', 'Part #': 'Part',
    'Quantity': 'Quantity', 'Equipment': 'Equipment'
})
df_import = df_import.rename(columns={
    'ACCOUNT NO.': 'Account', 'Part No.': 'Part',
    'Quantity': 'Quantity', 'EQUIP': 'Equipment'
})

# Normalize types to strings for accurate matching
for df in [df_master, df_import]:
    df['Account'] = df['Account'].astype(str).str.strip()
    df['Part'] = df['Part'].astype(str).str.strip()
    # Normalize Equipment if it exists in the dataframe
    if 'Equipment' in df.columns:
        df['Equipment'] = df['Equipment'].astype(str).str.strip()
    
# Master PO normalization
if 'PO' in df_master.columns:
    df_master['PO'] = df_master['PO'].astype(str).str.strip()

# 1. Summarize Equipment Import (Single line per Account/Part)
df_import_agg = df_import.groupby(['Account', 'Part']).agg({
    'Quantity': 'sum',
    'Equipment': 'first'
}).reset_index().rename(columns={'Quantity': 'Import_Qty'})

# 2. Summarize Master File for Comparison
df_master_agg = df_master.groupby(['Account', 'Part']).agg({
    'Quantity': 'sum'
}).reset_index().rename(columns={'Quantity': 'Master_Qty'})

# 3. Reconcile
df_reconcile = pd.merge(df_master_agg, df_import_agg, on=['Account', 'Part'], how='outer')
df_reconcile['Master_Qty'] = df_reconcile['Master_Qty'].fillna(0)
df_reconcile['Import_Qty'] = df_reconcile['Import_Qty'].fillna(0)
df_reconcile['Error'] = df_reconcile['Master_Qty'] != df_reconcile['Import_Qty']

# 4. Generate Valid Output
valid_combinations = df_reconcile[df_reconcile['Error'] == False][['Account', 'Part']]
df_no_error = pd.merge(df_master, valid_combinations, on=['Account', 'Part'], how='inner')
df_no_error = df_no_error[['PO', 'Account', 'Equipment', 'Quantity', 'Part']]

# 5. Generate Error Output
error_combinations = df_reconcile[df_reconcile['Error'] == True]

# Part A: Errors present in Master (Mismatch)
error_combinations_clean = error_combinations.drop(columns=['Equipment'], errors='ignore')
error_in_master = pd.merge(df_master, error_combinations_clean, on=['Account', 'Part'], how='inner')
error_in_master['Agg_Master_Qty'] = error_in_master['Master_Qty']
error_in_master['Agg_Import_Qty'] = error_in_master['Import_Qty']
error_in_master['Error'] = True

# Part B: Errors missing in Master (Present only in Import)
missing_in_master = error_combinations[error_combinations['Master_Qty'] == 0].copy()
missing_in_master['PO'] = "MISSING"
missing_in_master['Quantity'] = missing_in_master['Import_Qty']
missing_in_master['Agg_Master_Qty'] = 0
missing_in_master['Agg_Import_Qty'] = missing_in_master['Import_Qty']
missing_in_master['Error'] = True

# Combine and Format
cols_order = ['PO', 'Account', 'Equipment', 'Quantity', 'Part', 'Error', 'Agg_Master_Qty', 'Agg_Import_Qty']
# Ensure columns exist before reindexing
for col in cols_order:
    if col not in error_in_master.columns:
        error_in_master[col] = np.nan
    if col not in missing_in_master.columns:
        missing_in_master[col] = np.nan

df_error_final = pd.concat([
    error_in_master[cols_order], 
    missing_in_master[cols_order]
], ignore_index=True)

# Save output to the same folder as the Master file
import os
output_dir = os.path.dirname(master_path)
df_no_error.to_csv(os.path.join(output_dir, "Processed_Equipment_Valid.csv"), index=False)
df_error_final.to_csv(os.path.join(output_dir, "Processed_Equipment_Errors.csv"), index=False)

print("Done! Files saved to:", output_dir)

Please select the Master File (PO Part Accounts)...
Please select the Equipment Import File (Multiple Occurrences)...

Processing:
Master: C:/Users/ipalacio/Documents/000-LOG-AutomationFiles/EquipmentPOPartAccount_MasterFile.xlsx
Import: C:/Users/ipalacio/Documents/000-LOG-AutomationFiles/UCSD EQUIPMENT summary September'2025.xlsx



c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


KeyError: 'Account'